In [3]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
from itertools import product

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier , HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

import joblib

In [4]:
# ============================================================
# 2. SET PATHS
# ============================================================
DATA_PATH = Path(r"D:\khoane\MAHE\Project\dataset\UniPro\category_b_outputs\t2d_protein_classification_dataset_balanced_primary_only_v1.csv")
BASE_DIR = Path(r"D:\khoane\MAHE\Project\model\phase1_protein_baseline")
SPLIT_DIR = BASE_DIR / "splits"
FEATURE_DIR = BASE_DIR / "features"
RESULT_DIR = BASE_DIR / "results"
MODEL_DIR = BASE_DIR / "models"

for folder in [SPLIT_DIR, FEATURE_DIR, RESULT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# 2. LIGHTGBM IMPORT WITH FALLBACK
# ============================================================

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    print("LightGBM is available.")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM is not installed. Using HistGradientBoostingClassifier as fallback.")


LightGBM is available.


In [6]:
train_data = pd.read_csv(FEATURE_DIR / "protein_features_full_train_v1.csv")
val_data = pd.read_csv(FEATURE_DIR / "protein_features_full_val_v1.csv")
test_data = pd.read_csv(FEATURE_DIR / "protein_features_full_test_v1.csv")

In [7]:
metadata_and_label_cols = ["gene_id", "gene_symbol", "uniprot_accession", "sequence_clean_length", "label"]
# 3. Tách X (đặc trưng) và y (nhãn) cho tập TRAIN
X_train = train_data.drop(columns=metadata_and_label_cols)
y_train = train_data["label"]

# 4. Tách X (đặc trưng) và y (nhãn) cho tập VALIDATION
X_val = val_data.drop(columns=metadata_and_label_cols)
y_val = val_data["label"]

# 5. Tách X (đặc trưng) và y (nhãn) cho tập TEST
X_test = test_data.drop(columns=metadata_and_label_cols)
y_test = test_data["label"]

In [8]:
# ============================================================
# 4. BASIC SHAPE CHECK
# ============================================================

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\nTrain label distribution:")
print(pd.Series(y_train).value_counts())

print("\nValidation label distribution:")
print(pd.Series(y_val).value_counts())

print("\nTest label distribution:")
print(pd.Series(y_test).value_counts())

X_train: (1274, 427)
X_val: (273, 427)
X_test: (273, 427)
y_train: (1274,)
y_val: (273,)
y_test: (273,)

Train label distribution:
label
0    637
1    637
Name: count, dtype: int64

Validation label distribution:
label
1    137
0    136
Name: count, dtype: int64

Test label distribution:
label
0    137
1    136
Name: count, dtype: int64


In [9]:
# ============================================================
# 5. HELPER: GET CONTINUOUS SCORE FOR ROC-AUC / PR-AUC
# ============================================================

def get_positive_class_score(model, X):
    """
    Return continuous score for binary classification.

    Priority:
    1. predict_proba[:, 1]
    2. decision_function
    3. predict
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X)

    return model.predict(X)


# ============================================================
# 6. HELPER: EVALUATE BINARY CLASSIFIER
# ============================================================

def evaluate_binary_classifier(model, X, y, model_name, dataset_name):
    """
    Evaluate a trained binary classifier.
    """
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    result = {
        "model": model_name,
        "dataset": dataset_name,

        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),

        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return result

In [10]:
# ============================================================
# 7. DEFINE STRATIFIED 5-FOLD CV
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 8. DEFINE SCORING METRICS
# ============================================================

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [11]:
# ============================================================
# 9. DEFINE BASELINE PIPELINES
# ============================================================

baseline_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}


if LIGHTGBM_AVAILABLE:
    baseline_models["LightGBM"] = Pipeline([
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=15,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ))
    ])
else:
    baseline_models["Hist Gradient Boosting"] = Pipeline([
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            random_state=42
        ))
    ])

In [12]:
# ============================================================
# 10. RUN BASELINE 5-FOLD CV
# ============================================================

baseline_cv_results = []

for model_name, pipeline in baseline_models.items():
    print("=" * 80)
    print("Baseline CV:", model_name)

    cv_output = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )

    row = {
        "model": model_name,

        "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
        "train_roc_auc_std": cv_output["train_roc_auc"].std(),

        "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
        "cv_roc_auc_std": cv_output["test_roc_auc"].std(),

        "cv_f1_mean": cv_output["test_f1"].mean(),
        "cv_f1_std": cv_output["test_f1"].std(),

        "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
        "cv_pr_auc_std": cv_output["test_pr_auc"].std(),

        "cv_mcc_mean": cv_output["test_mcc"].mean(),
        "cv_mcc_std": cv_output["test_mcc"].std(),
    }

    baseline_cv_results.append(row)

baseline_cv_df = pd.DataFrame(baseline_cv_results).sort_values(
    by="cv_roc_auc_mean",
    ascending=False
)

display(baseline_cv_df)

baseline_cv_df.to_csv(
    RESULT_DIR / "baseline_5fold_cv_results_v1.csv",
    index=False
)

Baseline CV: Logistic Regression
Baseline CV: SVM RBF
Baseline CV: Random Forest
Baseline CV: LightGBM


,model,train_roc_auc_mean,train_roc_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_f1_std,cv_pr_auc_mean,cv_pr_auc_std,cv_mcc_mean,cv_mcc_std
2,Random Forest,1.000000,4.965068e-17,0.657308,0.045101,0.611924,0.039720,0.647828,0.044942,0.210841,0.078221
1,SVM RBF,0.971802,1.077056e-03,0.642960,0.039173,0.600954,0.028620,0.624027,0.031912,0.212736,0.061899
3,LightGBM,1.000000,0.000000e+00,0.616276,0.036255,0.573438,0.034731,0.614864,0.038853,0.133853,0.078715
0,Logistic Regression,0.906010,6.204817e-03,0.565644,0.015863,0.551586,0.027068,0.568392,0.017818,0.119769,0.056736


In [ ]:
# ============================================================
# 11. DEFINE SMALL PARAMETER GRIDS
# ============================================================
param_grids = {
    "Logistic Regression": {
        "model__C": [0.001, 0.003, 0.01, 0.03, 0.1, 1],
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"]
    },

    "SVM RBF": {
        "model__C": [0.01, 0.1, 1],
        "model__gamma": [0.0001, 0.001, 0.01, "scale"]
    },

    "Random Forest": {
        "model__n_estimators": [300, 500],
        "model__max_depth": [3, 5, 8, 10],
        "model__min_samples_leaf": [5, 10, 20],
        "model__max_features": ["sqrt", "log2", 0.2],
        "model__bootstrap": [True]
    },

    "LightGBM": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__num_leaves": [3, 7, 15],
        "model__max_depth": [2, 3, 5],
        "model__min_child_samples": [20, 50, 100],
        "model__subsample": [0.8],
        "model__colsample_bytree": [0.8],
        "model__reg_alpha": [0, 0.1],
        "model__reg_lambda": [0.1, 1]
    }
}

In [14]:
# ============================================================
# 12. RUN GRIDSEARCHCV
# ============================================================

grid_search_results = []
best_estimators = {}

for model_name, pipeline in baseline_models.items():
    print("=" * 80)
    print("GridSearchCV:", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_estimators[model_name] = grid.best_estimator_

    result_row = {
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_params": grid.best_params_
    }

    # Extract train score for best index
    best_idx = grid.best_index_
    result_row["best_train_roc_auc"] = grid.cv_results_["mean_train_score"][best_idx]
    result_row["overfit_gap_train_minus_cv"] = (
        result_row["best_train_roc_auc"] - result_row["best_cv_roc_auc"]
    )

    grid_search_results.append(result_row)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best params:", grid.best_params_)

grid_results_df = pd.DataFrame(grid_search_results).sort_values(
    by="best_cv_roc_auc",
    ascending=False
)

display(grid_results_df)

grid_results_df.to_csv(
    RESULT_DIR / "gridsearch_best_results_v1.csv",
    index=False
)

GridSearchCV: Logistic Regression
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV ROC-AUC: 0.6308287866575734
Best params: {'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
GridSearchCV: SVM RBF
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV ROC-AUC: 0.642982360964722
Best params: {'model__C': 1, 'model__gamma': 0.001}
GridSearchCV: Random Forest
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best CV ROC-AUC: 0.6499148467046935
Best params: {'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
GridSearchCV: LightGBM
Fitting 5 folds for each of 972 candidates, totalling 4860 fits
Best CV ROC-AUC: 0.6339256928513857
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_samples': 20, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model

,model,best_cv_roc_auc,best_params,best_train_roc_auc,overfit_gap_train_minus_cv
2,Random Forest,0.649915,"{'model__bootstrap': True, 'model__max_depth':...",0.998864,0.348949
1,SVM RBF,0.642982,"{'model__C': 1, 'model__gamma': 0.001}",0.894585,0.251603
3,LightGBM,0.633926,"{'model__colsample_bytree': 0.8, 'model__learn...",0.984435,0.350509
0,Logistic Regression,0.630829,"{'model__C': 0.001, 'model__penalty': 'l2', 'm...",0.792608,0.161780


In [15]:
# ============================================================
# 13. EVALUATE BEST TUNED MODELS ON TRAIN + VALIDATION
# ============================================================

tuned_eval_records = []

for model_name, model in best_estimators.items():
    print("=" * 80)
    print("Evaluating tuned model:", model_name)

    train_metrics = evaluate_binary_classifier(
        model=model,
        X=X_train,
        y=y_train,
        model_name=model_name,
        dataset_name="train"
    )

    val_metrics = evaluate_binary_classifier(
        model=model,
        X=X_val,
        y=y_val,
        model_name=model_name,
        dataset_name="validation"
    )

    tuned_eval_records.extend([train_metrics, val_metrics])

tuned_eval_df = pd.DataFrame(tuned_eval_records)

display(tuned_eval_df)

tuned_eval_df.to_csv(
    RESULT_DIR / "tuned_models_train_validation_metrics_v1.csv",
    index=False
)


# Validation-only comparison
validation_comparison = tuned_eval_df[
    tuned_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(validation_comparison)

validation_comparison.to_csv(
    RESULT_DIR / "tuned_models_validation_comparison_v1.csv",
    index=False
)

Evaluating tuned model: Logistic Regression
Evaluating tuned model: SVM RBF
Evaluating tuned model: Random Forest
Evaluating tuned model: LightGBM


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Logistic Regression,train,0.708006,0.705426,0.714286,0.701727,0.709828,0.780092,0.769753,0.416045,447,190,182,455
1,Logistic Regression,validation,0.582418,0.592000,0.540146,0.625000,0.564885,0.648884,0.648105,0.165734,85,51,63,74
2,SVM RBF,train,0.811617,0.796712,0.836735,0.786499,0.816233,0.884910,0.861855,0.624022,501,136,104,533
3,SVM RBF,validation,0.593407,0.601562,0.562044,0.625000,0.581132,0.647864,0.659846,0.187406,85,51,60,77
4,Random Forest,train,0.981162,0.982677,0.979592,0.982732,0.981132,0.997925,0.998249,0.962328,626,11,13,624
5,Random Forest,validation,0.615385,0.619403,0.605839,0.625000,0.612546,0.662838,0.671426,0.230877,85,51,54,83
6,LightGBM,train,0.913658,0.911076,0.916797,0.910518,0.913928,0.971486,0.973754,0.827332,580,57,53,584
7,LightGBM,validation,0.582418,0.589147,0.554745,0.610294,0.571429,0.640779,0.658528,0.165287,83,53,61,76


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,Random Forest,validation,0.615385,0.619403,0.605839,0.625000,0.612546,0.662838,0.671426,0.230877,85,51,54,83
1,Logistic Regression,validation,0.582418,0.592000,0.540146,0.625000,0.564885,0.648884,0.648105,0.165734,85,51,63,74
3,SVM RBF,validation,0.593407,0.601562,0.562044,0.625000,0.581132,0.647864,0.659846,0.187406,85,51,60,77
7,LightGBM,validation,0.582418,0.589147,0.554745,0.610294,0.571429,0.640779,0.658528,0.165287,83,53,61,76


In [16]:
pd.set_option("display.max_colwidth", None)

for _, row in grid_results_df.iterrows():
    print("=" * 80)
    print(row["model"])
    print("Best CV ROC-AUC:", row["best_cv_roc_auc"])
    print("Best Train ROC-AUC:", row["best_train_roc_auc"])
    print("Gap:", row["overfit_gap_train_minus_cv"])
    print("Best params:")
    print(row["best_params"])

Random Forest
Best CV ROC-AUC: 0.6499148467046935
Best Train ROC-AUC: 0.9988637542308421
Gap: 0.3489489075261486
Best params:
{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}
SVM RBF
Best CV ROC-AUC: 0.642982360964722
Best Train ROC-AUC: 0.8945852592587558
Gap: 0.2516028982940337
Best params:
{'model__C': 1, 'model__gamma': 0.001}
LightGBM
Best CV ROC-AUC: 0.6339256928513857
Best Train ROC-AUC: 0.9844345208016563
Gap: 0.3505088279502706
Best params:
{'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_child_samples': 20, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__reg_alpha': 0.1, 'model__reg_lambda': 0.1, 'model__subsample': 0.8}
Logistic Regression
Best CV ROC-AUC: 0.6308287866575734
Best Train ROC-AUC: 0.7926084995267801
Gap: 0.16177971286920678
Best params:
{'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


In [28]:
# ============================================================
# 1. BUILD SELECTED ESTIMATORS FOR ENSEMBLE
# ============================================================

selected_for_ensemble = {
    "Logistic Regression": best_estimators["Logistic Regression"],
    "SVM RBF": best_estimators["SVM RBF"],
    "Random Forest": best_estimators["Random Forest"],
}

# Optional: only add LightGBM later if you want to compare
# selected_for_ensemble["LightGBM"] = best_estimators["LightGBM"]


# ============================================================
# 2. SOFT VOTING CLASSIFIER
# ============================================================

from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

voting_estimators = [
    ("logreg", selected_for_ensemble["Logistic Regression"]),
    ("svm_rbf", selected_for_ensemble["SVM RBF"]),
    ("rf", selected_for_ensemble["Random Forest"]),
]

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=-1
)

soft_voting.fit(X_train, y_train)

voting_train_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_train,
    y=y_train,
    model_name="Soft Voting LR+SVM+RF",
    dataset_name="train"
)

voting_val_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_val,
    y=y_val,
    model_name="Soft Voting LR+SVM+RF",
    dataset_name="validation"
)

display(pd.DataFrame([voting_train_metrics, voting_val_metrics]))

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Soft Voting LR+SVM+RF,train,0.864992,0.861586,0.869702,0.860283,0.865625,0.934576,0.935256,0.730017,548,89,83,554
1,Soft Voting LR+SVM+RF,validation,0.611722,0.626016,0.562044,0.661765,0.592308,0.657900,0.667151,0.224910,90,46,60,77


In [26]:
# ============================================================
# 14. BUILD SOFT VOTING CLASSIFIER
# ============================================================

voting_estimators = []

for model_name, estimator in best_estimators.items():
    short_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    voting_estimators.append((short_name, estimator))

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=-1
)

print("Training Soft Voting Classifier...")
soft_voting.fit(X_train, y_train)

best_estimators["Soft Voting"] = soft_voting

voting_train_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_train,
    y=y_train,
    model_name="Soft Voting",
    dataset_name="train"
)

voting_val_metrics = evaluate_binary_classifier(
    model=soft_voting,
    X=X_val,
    y=y_val,
    model_name="Soft Voting",
    dataset_name="validation"
)

display(pd.DataFrame([voting_train_metrics, voting_val_metrics]))

Training Soft Voting Classifier...


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Soft Voting,train,0.890895,0.881902,0.902669,0.879121,0.892164,0.958437,0.960375,0.782006,560,77,62,575
1,Soft Voting,validation,0.608059,0.619048,0.569343,0.647059,0.593156,0.660208,0.678238,0.217044,88,48,59,78


In [25]:
# ============================================================
# 15. BUILD STACKING CLASSIFIER
# ============================================================

stacking_estimators = []

for model_name, estimator in best_estimators.items():
    if model_name in ["Soft Voting"]:
        continue

    short_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    stacking_estimators.append((short_name, estimator))


stacking = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        random_state=42
    ),
    cv=5,
    stack_method="auto",
    n_jobs=-1,
    passthrough=False
)

print("Training Stacking Classifier...")
stacking.fit(X_train, y_train)

best_estimators["Stacking"] = stacking

stacking_train_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_train,
    y=y_train,
    model_name="Stacking",
    dataset_name="train"
)

stacking_val_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_val,
    y=y_val,
    model_name="Stacking",
    dataset_name="validation"
)

display(pd.DataFrame([stacking_train_metrics, stacking_val_metrics]))

Training Stacking Classifier...


,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Stacking,train,0.909733,0.902778,0.918367,0.901099,0.910506,0.969184,0.970924,0.819588,574,63,52,585
1,Stacking,validation,0.604396,0.616000,0.562044,0.647059,0.587786,0.658705,0.677789,0.209847,88,48,60,77


In [29]:
# ============================================================
# 3. STACKING CLASSIFIER
# ============================================================

stacking_estimators = [
    ("logreg", selected_for_ensemble["Logistic Regression"]),
    ("svm_rbf", selected_for_ensemble["SVM RBF"]),
    ("rf", selected_for_ensemble["Random Forest"]),
]

stacking = StackingClassifier(
    estimators=stacking_estimators,
    final_estimator=LogisticRegression(
        max_iter=5000,
        C=1.0,
        random_state=42
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

stacking.fit(X_train, y_train)

stacking_train_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_train,
    y=y_train,
    model_name="Stacking LR+SVM+RF",
    dataset_name="train"
)

stacking_val_metrics = evaluate_binary_classifier(
    model=stacking,
    X=X_val,
    y=y_val,
    model_name="Stacking LR+SVM+RF",
    dataset_name="validation"
)

display(pd.DataFrame([stacking_train_metrics, stacking_val_metrics]))

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Stacking LR+SVM+RF,train,0.885400,0.878274,0.894819,0.875981,0.886470,0.957454,0.958570,0.770937,558,79,67,570
1,Stacking LR+SVM+RF,validation,0.611722,0.626016,0.562044,0.661765,0.592308,0.659886,0.674216,0.224910,90,46,60,77


In [30]:
# ============================================================
# 4. COMBINE ALL MODEL RESULTS
# ============================================================

ensemble_records = [
    voting_train_metrics,
    voting_val_metrics,
    stacking_train_metrics,
    stacking_val_metrics
]

all_eval_df = pd.concat(
    [
        tuned_eval_df,
        pd.DataFrame(ensemble_records)
    ],
    ignore_index=True
)

validation_final_comparison = all_eval_df[
    all_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(validation_final_comparison)

overfit_final_table = all_eval_df.pivot_table(
    index="model",
    columns="dataset",
    values=["roc_auc", "f1", "mcc"]
)

display(overfit_final_table)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,Random Forest,validation,0.615385,0.619403,0.605839,0.625000,0.612546,0.662838,0.671426,0.230877,85,51,54,83
11,Stacking LR+SVM+RF,validation,0.611722,0.626016,0.562044,0.661765,0.592308,0.659886,0.674216,0.224910,90,46,60,77
9,Soft Voting LR+SVM+RF,validation,0.611722,0.626016,0.562044,0.661765,0.592308,0.657900,0.667151,0.224910,90,46,60,77
1,Logistic Regression,validation,0.582418,0.592000,0.540146,0.625000,0.564885,0.648884,0.648105,0.165734,85,51,63,74
3,SVM RBF,validation,0.593407,0.601562,0.562044,0.625000,0.581132,0.647864,0.659846,0.187406,85,51,60,77
7,LightGBM,validation,0.582418,0.589147,0.554745,0.610294,0.571429,0.640779,0.658528,0.165287,83,53,61,76


f1                  mcc              roc_auc  \
dataset                   train validation     train validation     train   
model                                                                       
LightGBM               0.913928   0.571429  0.827332   0.165287  0.971486   
Logistic Regression    0.709828   0.564885  0.416045   0.165734  0.780092   
Random Forest          0.981132   0.612546  0.962328   0.230877  0.997925   
SVM RBF                0.816233   0.581132  0.624022   0.187406  0.884910   
Soft Voting LR+SVM+RF  0.865625   0.592308  0.730017   0.224910  0.934576   
Stacking LR+SVM+RF     0.886470   0.592308  0.770937   0.224910  0.957454   

                                  
dataset               validation  
model                             
LightGBM                0.640779  
Logistic Regression     0.648884  
Random Forest           0.662838  
SVM RBF                 0.647864  
Soft Voting LR+SVM+RF   0.657900  
Stacking LR+SVM+RF      0.659886

In [27]:
# ============================================================
# 16. COMBINE INDIVIDUAL + ENSEMBLE VALIDATION RESULTS
# ============================================================

all_model_eval_records = tuned_eval_records.copy()

all_model_eval_records.extend([
    voting_train_metrics,
    voting_val_metrics,
    stacking_train_metrics,
    stacking_val_metrics
])

all_model_eval_df = pd.DataFrame(all_model_eval_records)

all_validation_results = all_model_eval_df[
    all_model_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(all_validation_results)

all_model_eval_df.to_csv(
    RESULT_DIR / "all_models_train_validation_metrics_v1.csv",
    index=False
)

all_validation_results.to_csv(
    RESULT_DIR / "all_models_validation_comparison_v1.csv",
    index=False
)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,Random Forest,validation,0.615385,0.619403,0.605839,0.625000,0.612546,0.662838,0.671426,0.230877,85,51,54,83
9,Soft Voting,validation,0.608059,0.619048,0.569343,0.647059,0.593156,0.660208,0.678238,0.217044,88,48,59,78
11,Stacking,validation,0.604396,0.616000,0.562044,0.647059,0.587786,0.658705,0.677789,0.209847,88,48,60,77
1,Logistic Regression,validation,0.582418,0.592000,0.540146,0.625000,0.564885,0.648884,0.648105,0.165734,85,51,63,74
3,SVM RBF,validation,0.593407,0.601562,0.562044,0.625000,0.581132,0.647864,0.659846,0.187406,85,51,60,77
7,LightGBM,validation,0.582418,0.589147,0.554745,0.610294,0.571429,0.640779,0.658528,0.165287,83,53,61,76


In [ ]:
# ============================================================
# 17. SELECT BEST MODEL BASED ON VALIDATION PERFORMANCE
# ============================================================

best_validation_row = all_validation_results.iloc[0]
best_model_name = best_validation_row["model"]
best_model = best_estimators[best_model_name]

print("Best model selected from validation set:")
print(best_model_name)
display(best_validation_row)

pd.DataFrame([best_validation_row]).to_csv(
    RESULT_DIR / "best_model_validation_summary_v1.csv",
    index=False
)

In [ ]:
# ============================================================
# 18. FINAL TEST EVALUATION
# ============================================================

test_metrics = evaluate_binary_classifier(
    model=best_model,
    X=X_test,
    y=y_test,
    model_name=best_model_name,
    dataset_name="test"
)

test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)

test_metrics_df.to_csv(
    RESULT_DIR / "final_test_metrics_v1.csv",
    index=False
)

In [ ]:
# ============================================================
# 19. SAVE ALL BEST ESTIMATORS
# ============================================================

for model_name, model in best_estimators.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = MODEL_DIR / f"{safe_name}_best_estimator_v1.pkl"
    joblib.dump(model, model_path)

    print("Saved:", model_path)


# Save final selected model separately
final_model_path = MODEL_DIR / "final_selected_model_v1.pkl"
joblib.dump(best_model, final_model_path)

print("Final selected model saved to:", final_model_path)

In [ ]:
# ============================================================
# 21. SAVE FINAL TEST PREDICTIONS
# ============================================================

y_test_score = get_positive_class_score(best_model, X_test)
y_test_pred = best_model.predict(X_test)

test_predictions_df = pd.DataFrame({
    "true_label": y_test.values,
    "pred_label": y_test_pred,
    "pred_score_t2d_associated": y_test_score
})

# Optional: attach metadata if test_df exists
if "test_df" in globals():
    metadata_cols = [
        col for col in [
            "gene_id",
            "gene_symbol",
            "uniprot_accession",
            "sequence_length_fasta"
        ]
        if col in test_df.columns
    ]

    metadata_df = test_df[metadata_cols].reset_index(drop=True)
    test_predictions_df = pd.concat(
        [metadata_df, test_predictions_df.reset_index(drop=True)],
        axis=1
    )

display(test_predictions_df.head())

test_predictions_df.to_csv(
    RESULT_DIR / "final_test_predictions_v1.csv",
    index=False
)

In [ ]:
# ============================================================
# 22. OVERFITTING CHECK TABLE
# ============================================================

overfit_table = all_model_eval_df.pivot_table(
    index="model",
    columns="dataset",
    values=["roc_auc", "f1", "mcc"]
)

display(overfit_table)

overfit_table.to_csv(
    RESULT_DIR / "all_models_overfitting_check_v1.csv"
)

In [ ]:
# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("=" * 80)
print("PHASE 1.1 MODEL TRAINING COMPLETE")
print("=" * 80)

print("Best model selected on validation set:", best_model_name)

print("\nValidation performance:")
display(pd.DataFrame([best_validation_row]))

print("\nFinal test performance:")
display(test_metrics_df)

print("\nSaved results to:", RESULT_DIR)
print("Saved models to:", MODEL_DIR)

In [31]:
# ============================================================
# FINAL MODEL SELECTION BASED ON VALIDATION SET
# ============================================================

final_model_name = "Random Forest"
final_model = best_estimators[final_model_name]

print("Final selected model:", final_model_name)

Final selected model: Random Forest


In [32]:
# ============================================================
# FINAL TEST EVALUATION
# Use test set only once after final model is selected
# ============================================================

final_test_metrics = evaluate_binary_classifier(
    model=final_model,
    X=X_test,
    y=y_test,
    model_name=final_model_name,
    dataset_name="test"
)

final_test_metrics_df = pd.DataFrame([final_test_metrics])

display(final_test_metrics_df)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Random Forest,test,0.567766,0.56338,0.588235,0.547445,0.57554,0.579755,0.561841,0.13579,75,62,56,80


In [33]:
# ============================================================
# FINAL TEST CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix

y_test_pred = final_model.predict(X_test)

cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_0_Background", "Actual_1_T2D"],
    columns=["Pred_0_Background", "Pred_1_T2D"]
)

display(cm_df)

,Pred_0_Background,Pred_1_T2D
Actual_0_Background,75,62
Actual_1_T2D,56,80


In [35]:
# ============================================================
# FINAL TEST PREDICTIONS
# ============================================================

y_test_score = get_positive_class_score(final_model, X_test)
y_test_pred = final_model.predict(X_test)

test_predictions_df = pd.DataFrame({
    "true_label": y_test.values,
    "pred_label": y_test_pred,
    "pred_score_t2d_associated": y_test_score
})

# Attach metadata if test_df exists
metadata_cols = [
    col for col in [
        "gene_id",
        "gene_symbol",
        "uniprot_accession",
        "sequence_length_fasta"
    ]
    if col in test_data.columns
]

if len(metadata_cols) > 0:
    metadata_df = test_data[metadata_cols].reset_index(drop=True)
    test_predictions_df = pd.concat(
        [metadata_df, test_predictions_df.reset_index(drop=True)],
        axis=1
    )

display(test_predictions_df.head())

,gene_id,gene_symbol,uniprot_accession,true_label,pred_label,pred_score_t2d_associated
0,ENSG00000177542,SLC25A22,Q9H936,0,0,0.475793
1,ENSG00000123080,CDKN2C,P42773,1,0,0.463354
2,ENSG00000185262,UBALD2,Q8IYN6,0,1,0.529447
3,ENSG00000092148,HECTD1,Q9ULT8,0,1,0.530754
4,ENSG00000139364,TMEM132B,Q14DG7,1,1,0.562602


In [36]:
# ============================================================
# SAVE FINAL OUTPUTS
# ============================================================

from pathlib import Path
import joblib

RESULT_DIR = Path("../modeling/phase1_protein_baseline/results")
MODEL_DIR = Path("../modeling/phase1_protein_baseline/models")

RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

final_test_metrics_df.to_csv(
    RESULT_DIR / "final_test_metrics_random_forest_v1.csv",
    index=False
)

cm_df.to_csv(
    RESULT_DIR / "final_test_confusion_matrix_random_forest_v1.csv"
)

test_predictions_df.to_csv(
    RESULT_DIR / "final_test_predictions_random_forest_v1.csv",
    index=False
)

joblib.dump(
    final_model,
    MODEL_DIR / "final_random_forest_protein_baseline_v1.pkl"
)

print("Final outputs saved.")

Final outputs saved.


In [37]:
# ============================================================
# DIAGNOSTIC ONLY: TEST ALL TUNED MODELS + ENSEMBLES
# Do not use this to re-select the final model in a strict protocol.
# ============================================================

diagnostic_models = {}

# Tuned individual models
diagnostic_models.update(best_estimators)

# If you have these variables from ensemble stage
if "soft_voting" in globals():
    diagnostic_models["Soft Voting LR+SVM+RF"] = soft_voting

if "stacking" in globals():
    diagnostic_models["Stacking LR+SVM+RF"] = stacking

diagnostic_test_records = []

for model_name, model in diagnostic_models.items():
    metrics = evaluate_binary_classifier(
        model=model,
        X=X_test,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic"
    )
    diagnostic_test_records.append(metrics)

diagnostic_test_df = pd.DataFrame(diagnostic_test_records).sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

display(diagnostic_test_df)

,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
3,LightGBM,test_diagnostic,0.593407,0.582781,0.647059,0.540146,0.613240,0.605571,0.584510,0.188269,74,63,48,88
4,Soft Voting,test_diagnostic,0.556777,0.552448,0.580882,0.532847,0.566308,0.601599,0.580909,0.113857,73,64,57,79
5,Stacking,test_diagnostic,0.564103,0.560284,0.580882,0.547445,0.570397,0.600633,0.584769,0.128397,75,62,57,79
0,Logistic Regression,test_diagnostic,0.556777,0.553957,0.566176,0.547445,0.560000,0.597735,0.573440,0.113640,75,62,59,77
1,SVM RBF,test_diagnostic,0.542125,0.540146,0.544118,0.540146,0.542125,0.597386,0.588033,0.084264,74,63,62,74
6,Soft Voting LR+SVM+RF,test_diagnostic,0.542125,0.539568,0.551471,0.532847,0.545455,0.592958,0.580909,0.084331,73,64,61,75
7,Stacking LR+SVM+RF,test_diagnostic,0.553114,0.552239,0.544118,0.562044,0.548148,0.592261,0.584924,0.106179,77,60,62,74
2,Random Forest,test_diagnostic,0.567766,0.563380,0.588235,0.547445,0.575540,0.579755,0.561841,0.135790,75,62,56,80


In [38]:
# ============================================================
# FEATURE SET ABLATION SETUP
# ============================================================

length_cols = ["protein_length"]
aac_cols = [c for c in X_train.columns if c.startswith("AAC_")]
dpc_cols = [c for c in X_train.columns if c.startswith("DPC_")]
physchem_cols = [
    c for c in X_train.columns
    if c.startswith("frac_") or c == "charge_balance"
]

feature_sets = {
    "Length only": length_cols,
    "AAC only": aac_cols,
    "AAC + Physchem": aac_cols + physchem_cols,
    "AAC + DPC": aac_cols + dpc_cols,
    "Full": length_cols + aac_cols + dpc_cols + physchem_cols
}

for name, cols in feature_sets.items():
    print(name, len(cols))

Length only 1
AAC only 20
AAC + Physchem 26
AAC + DPC 420
Full 427


In [39]:
# ============================================================
# ABLATION: LOGISTIC REGRESSION + RANDOM FOREST
# ============================================================

from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

ablation_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=0.001,
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            max_features="sqrt",
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

ablation_records = []

for feature_set_name, cols in feature_sets.items():
    Xtr_sub = X_train[cols]

    for model_name, model in ablation_models.items():
        print("=" * 80)
        print(feature_set_name, "|", model_name)

        cv_output = cross_validate(
            estimator=model,
            X=Xtr_sub,
            y=y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=True
        )

        row = {
            "feature_set": feature_set_name,
            "n_features": len(cols),
            "model": model_name,
            "train_roc_auc_mean": cv_output["train_roc_auc"].mean(),
            "cv_roc_auc_mean": cv_output["test_roc_auc"].mean(),
            "cv_roc_auc_std": cv_output["test_roc_auc"].std(),
            "cv_f1_mean": cv_output["test_f1"].mean(),
            "cv_pr_auc_mean": cv_output["test_pr_auc"].mean(),
            "cv_mcc_mean": cv_output["test_mcc"].mean(),
            "overfit_gap": cv_output["train_roc_auc"].mean() - cv_output["test_roc_auc"].mean()
        }

        ablation_records.append(row)

ablation_cv_df = pd.DataFrame(ablation_records).sort_values(
    by=["cv_roc_auc_mean", "cv_mcc_mean"],
    ascending=False
)

display(ablation_cv_df)

Length only | Logistic Regression
Length only | Random Forest
AAC only | Logistic Regression
AAC only | Random Forest
AAC + Physchem | Logistic Regression
AAC + Physchem | Random Forest
AAC + DPC | Logistic Regression
AAC + DPC | Random Forest
Full | Logistic Regression
Full | Random Forest


,feature_set,n_features,model,train_roc_auc_mean,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_pr_auc_mean,cv_mcc_mean,overfit_gap
5,AAC + Physchem,26,Random Forest,0.944876,0.650436,0.048805,0.616848,0.649417,0.218550,0.294440
9,Full,427,Random Forest,0.998864,0.649915,0.034630,0.615495,0.637021,0.220113,0.348949
3,AAC only,20,Random Forest,0.940967,0.643354,0.052381,0.602763,0.640019,0.190167,0.297613
7,AAC + DPC,420,Random Forest,0.998957,0.641280,0.043342,0.590815,0.629810,0.184059,0.357678
6,AAC + DPC,420,Logistic Regression,0.793092,0.631112,0.033138,0.600989,0.610982,0.190317,0.161980
8,Full,427,Logistic Regression,0.792608,0.630829,0.034117,0.594712,0.610408,0.177747,0.161780
2,AAC only,20,Logistic Regression,0.639785,0.619368,0.040023,0.583935,0.597005,0.163757,0.020417
4,AAC + Physchem,26,Logistic Regression,0.636755,0.616484,0.038391,0.590235,0.593881,0.176457,0.020271
1,Length only,1,Random Forest,0.710575,0.510237,0.003517,0.516447,0.506585,0.026943,0.200338
0,Length only,1,Logistic Regression,0.503161,0.491936,0.023136,0.483975,0.501221,-0.006177,0.011225
